In [1]:
import pandas as pd
import numpy as np

In [2]:
#read the data
trade_2022 = pd.read_csv('/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Trade  data/baci data/2017-2022/BACI_HS17_Y2022_V202401b.csv',dtype = {'k':str})
country_data = pd.read_csv('/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Trade  data/baci data/2017-2022/country_codes_V202401b.csv')
product_data = pd.read_csv('/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Trade  data/baci data/2017-2022/product_codes_HS17_V202401b.csv',dtype = {'code':str})

In [3]:
# rename columns as per the documentation and to fit with the other dataframes
trade_2022 = trade_2022.rename(columns={
    't': 'period',
    'i': 'exporter',
    'j': 'importer',
    'k': 'cmdCode',
    'v': 'value',
    'q': 'quantity'
})

In [4]:
trade_2022.head()

,period,exporter,importer,cmdCode,value,quantity
0,2022,4,20,210610,0.393,0.002
1,2022,4,20,210690,0.067,0.001
2,2022,4,20,271000,6.329,8.103
3,2022,4,20,843131,0.346,0.022
4,2022,4,24,071332,35.529,25.060


In [5]:
# process Auxiliary dataframes
country_data = country_data.rename(columns={'country_code': 'exporter'})
country_data_copy = country_data.copy()
# for use in the importers
country_data_copy = country_data_copy.drop('country_iso2', axis=1)
country_data_copy = country_data_copy.rename(columns={'exporter': 'importer', 'country_iso3':'importerISO'})

In [6]:
trade_2022['HS4_code'] = trade_2022['cmdCode'].astype(str).str[:4]

In [7]:
trade_2022.head()

,period,exporter,importer,cmdCode,value,quantity,HS4_code
0,2022,4,20,210610,0.393,0.002,2106
1,2022,4,20,210690,0.067,0.001,2106
2,2022,4,20,271000,6.329,8.103,2710
3,2022,4,20,843131,0.346,0.022,8431
4,2022,4,24,071332,35.529,25.060,0713


In [8]:
grouped_df_2022 = trade_2022.groupby(['exporter', 'importer', 'HS4_code','cmdCode','period'], as_index=False)[['value']].sum()
processed_df_2022 = pd.merge(grouped_df_2022, country_data, on='exporter', how='inner')
Baci_2022  = pd.merge(processed_df_2022, country_data_copy, on='importer', how='inner')

In [9]:
Baci_2022.drop(['exporter','importer', 'country_iso2'], axis=1, inplace=True)
Baci_2022.rename(columns={'country_name_x': 'exporter', 'country_name_y': 'importer','cmdCode':'H0','country_iso3':'exporterISO', 'value':'reconciled_value'}, inplace=True)
Baci_2022['reconciled_value'] = Baci_2022['reconciled_value'] * 1000

In [11]:
Baci_2022.sort_values(by='HS4_code', inplace=True)

In [12]:
Baci_2022.head()

,HS4_code,H0,period,reconciled_value,exporter,exporterISO,importer,importerISO
461419,0101,010129,2022,335442.0,Bahrain,BHR,Germany,DEU
705398,0101,010190,2022,4780.0,Belgium,BEL,Netherlands,NLD
705397,0101,010129,2022,2351127.0,Belgium,BEL,Netherlands,NLD
705396,0101,010121,2022,2106292.0,Belgium,BEL,Netherlands,NLD
6699640,0101,010121,2022,5710036.0,Netherlands,NLD,United Kingdom,GBR


In [13]:
usa_df = Baci_2022[Baci_2022['importerISO'] == 'USA']

In [14]:
usa_df.head()

,HS4_code,H0,period,reconciled_value,exporter,exporterISO,importer,importerISO
3253111,0101,010129,2022,11105895.0,France,FRA,USA,USA
3253110,0101,010121,2022,19716067.0,France,FRA,USA,USA
6706299,0101,010121,2022,7631310.0,Netherlands,NLD,USA,USA
6706300,0101,010129,2022,122396966.0,Netherlands,NLD,USA,USA
10354045,0101,010130,2022,5504.0,Türkiye,TUR,USA,USA


In [17]:
usa_df.to_csv('/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Reconciled data/usa_2022.csv', index=False)

In [ ]:
# SAVE AS CSV FILE
root_folder = "/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation//Reconciled data"
file_name= "Baci_2021.csv"
Baci_2021.to_csv(f"{root_folder}/{file_name}", index=False)